# OurAirports Data Cleansing & Merge

**Purpose**: merge `airports-ourairports.csv` with `runways.csv` (the list attribute for requirement 6), do light cleaning, and save one integrated OurAirports-side dataset to `data/interim/`.

Unlike Wikidata, there's no query-reliability problem here (it's a flat CSV download), so there's no separate profiling notebook needed — this single notebook covers merge + cleaning.

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

RAW_DIR = project_root / "data" / "raw" / "ourairports"
INTERIM_DIR = project_root / "data" / "interim"

airports = pd.read_csv(RAW_DIR / "airports-ourairports.csv")
runways = pd.read_csv(RAW_DIR / "runways.csv")

print(f"airports: {airports.shape}")
print(f"runways:  {runways.shape}")

airports: (86151, 19)
runways:  (48278, 20)


## `type` overview

Not filtering out `closed` airports here -- that's a decision for the matching/fusion stage, not cleansing. Keeping the full set and the `type` column so it's available as a filter later.

In [2]:
airports["type"].value_counts()

type
small_airport     42762
heliport          23224
closed            13548
medium_airport     4106
seaplane_base      1273
large_airport      1175
balloonport          63
Name: count, dtype: int64

## Aggregate runways into a list attribute

Confirmed in `requirement_analysis.ipynb`: `runways.airport_ident` joins cleanly to `airports.ident` (no orphans), 41,163 airports have at least one runway, ~13% have more than one. Each airport gets a `runways` column holding a list of dicts (one per runway) instead of flattening into separate rows -- that's what makes this the list attribute for requirement 6.

In [3]:
RUNWAY_FIELDS = ["length_ft", "width_ft", "surface", "lighted", "closed", "le_ident", "he_ident"]

def runway_to_dict(row):
    return {field: row[field] for field in RUNWAY_FIELDS if pd.notna(row[field])}

runways_by_airport = (
    runways.apply(runway_to_dict, axis=1)
    .groupby(runways["airport_ident"])
    .apply(list)
    .rename("runways")
)

print(f"airports with runway data: {len(runways_by_airport):,}")
runways_by_airport.head(3)

airports with runway data: 41,163


airport_ident
00A     [{'length_ft': 80.0, 'width_ft': 80.0, 'surfac...
00AK    [{'length_ft': 2500.0, 'width_ft': 40.0, 'surf...
00AL    [{'length_ft': 2100.0, 'width_ft': 90.0, 'surf...
Name: runways, dtype: object

## Merge + trim irrelevant columns

Dropping only `keywords` -- free-text tags, high-noise, no clear integration value. Keeping `home_link` and, importantly, `wikipedia_link`: OurAirports links each airport directly to its Wikipedia article, and Wikidata items generally carry a Wikipedia sitelink (DBpedia resource URIs are literally derived from the article title) -- so this is a potential high-precision, near-exact matching key against Wikidata for identity resolution later. Dropping it earlier was a mistake.

In [4]:
DROP_COLUMNS = ["keywords"]

final_df = airports.drop(columns=DROP_COLUMNS).merge(
    runways_by_airport, left_on="ident", right_index=True, how="left"
)

print(final_df.shape)
print(f"airports with a non-empty runways list: {final_df['runways'].notna().sum():,}")
print(f"airports with a wikipedia_link: {final_df['wikipedia_link'].notna().sum():,} ({final_df['wikipedia_link'].notna().mean():.1%})")
final_df.head(3)

(86151, 19)
airports with a non-empty runways list: 41,163
airports with a wikipedia_link: 16,757 (19.5%)


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,runways
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,"[{'length_ft': 80.0, 'width_ft': 80.0, 'surfac..."
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,"[{'length_ft': 2500.0, 'width_ft': 40.0, 'surf..."


In [5]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
out_path = INTERIM_DIR / "ourairports_airports.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved {len(final_df):,} airports to {out_path}")

Saved 86,151 airports to /Users/huanglinchun/Desktop/Master/Studies/Web-Data-Integration/Web-Data-Integration-Project/data/interim/ourairports_airports.csv


## Notes

- `runways` is saved as a Python-list-of-dicts literal inside a CSV cell (via pandas' default `str()` repr) -- fine for now, but re-reading this CSV later needs `ast.literal_eval` on that column, not `pd.read_csv` alone. If this friction becomes annoying, switch to Parquet, which preserves nested types natively.
- `closed` airports (13,548 of them) are still in the dataset -- filtering, if wanted, belongs at the matching/fusion stage, not here.
- No profiling notebook for this source: it's a flat CSV, not a queried API, so there's no query-reliability story to document the way there was for Wikidata/QLever.
- `wikipedia_link` is kept (19.5% coverage, 16,757 airports) as a planned high-precision matching key against Wikidata during identity resolution 
- - OurAirports' data provenance is largely independent of Wikidata/DBpedia (mostly FAA/DAFIF + community + specialist aviation databases, per ourairports.com/about.html), so this isn't circular, it's just two independently-maintained sources that both happen to reference the same Wikipedia article for a given airport.